In [ ]:
## NLP PROJECT ON SOCIAL STIGMA DETECTION IN HEALTH DISCUSSION


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Install required libraries
# pip = Python's package installer
# -q means 'quiet' (less output spam)
# ─────────────────────────────────────────────────────────────────────────────

!pip install -q vaderSentiment           # For sentiment analysis
!pip install -q textblob                 # Another sentiment tool
!pip install -q scikit-learn             # For text classification (ML)
!pip install -q spacy                    # For NER and semantics
!python -m spacy download en_core_web_md -q  # spaCy English model (with word vectors)

print("\n✅ All libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 23.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

✅ All libraries installed successfully!


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Import everything we need
# Think of imports as 'loading tools into your toolbox'
# ─────────────────────────────────────────────────────────────────────────────

# --- Sentiment Analysis ---
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# --- Text Classification (Machine Learning) ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

# --- Semantics (Word Meanings) ---
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- Named Entity Recognition ---
import spacy

# --- Display / Visualization ---
import warnings
warnings.filterwarnings('ignore')  # Hide unimportant warnings

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [ ]:
# Step 1 DATASETS   ────────────────────────────────────────────────────────────
# Our sample health forum posts
# Each post has a 'text' (what someone wrote) and a 'label' (what it means)


posts = [
    # ── STIGMA posts ──────────────────────────────────────────────────────────
    {"text": "Schizophrenics are violent and dangerous. Keep them away from society.","label": "stigma"},
    {"text": "Drug addicts chose this lifestyle. They deserve no sympathy.","label": "stigma"},
    {"text": "People with depression are just weak. They need to man up.","label": "stigma"},
    {"text": "Mental illness patients are a burden on the healthcare system.","label": "stigma"},
    {"text": "She is crazy. I would never hire someone with bipolar disorder.","label": "stigma"},
    {"text": "Alcoholics are just selfish people who ruin their families.","label": "stigma"},
    {"text": "Why waste resources treating people who will never get better anyway?","label": "stigma"},

    # ── SUPPORT posts ─────────────────────────────────────────────────────────
    {"text": "Recovery is possible. You are not alone in this journey.","label": "support"},
    {"text": "People with mental health conditions deserve kindness and proper care.","label": "support"},
    {"text": "My friend has depression. I try to listen and support them every day.","label": "support"},
    {"text": "Therapy and medication together have helped many people with bipolar.","label": "support"},
    {"text": "Let us break the stigma around addiction and offer compassionate help.","label": "support"},
    {"text": "Mental health is just as important as physical health. Seek help early.","label": "support"},

    # ── INFO posts ────────────────────────────────────────────────────────────
    {"text": "Depression affects over 280 million people worldwide according to WHO.","label": "info"},
    {"text": "ADHD is a neurodevelopmental disorder affecting approximately 5% of children.","label": "info"},
    {"text": "Schizophrenia affects about 1 in 300 people globally.","label": "info"},
    {"text": "The clinic offers free mental health counselling on weekdays.","label": "info"},
    {"text": "Sertraline is a commonly prescribed SSRI for anxiety and depression.","label": "info"},
    {"text": "NIMHANS in Bengaluru is India's premier mental health institution.","label": "info"},
    {"text": "Cognitive Behavioural Therapy is an evidence-based treatment for OCD.","label": "info"},
]

# ─────────────────────────────────────────────────────────────────────────────
# Separate the text and labels into two lists
# This is standard practice in ML
# ─────────────────────────────────────────────────────────────────────────────
texts  = [p["text"]  for p in posts]   # List of all post texts
labels = [p["label"] for p in posts]   # List of all post labels

print(f"✅ Dataset ready! Total posts: {len(texts)}")
print(f"   Stigma posts  : {labels.count('stigma')}")
print(f"   Support posts : {labels.count('support')}")
print(f"   Info posts    : {labels.count('info')}")

✅ Dataset ready! Total posts: 20
   Stigma posts  : 7
   Support posts : 6
   Info posts    : 7


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SENTIMENT CLASSIFICATION using VADER
# ─────────────────────────────────────────────────────────────────────────────

# Step 1: Create the VADER analyser object
analyzer = SentimentIntensityAnalyzer()

# Step 2: Define a helper function to classify one post
# A 'function' is a reusable block of code — we call it later for every post
def get_sentiment(text):
    """
    Takes a text string and returns:
    - The sentiment label: POSITIVE / NEGATIVE / NEUTRAL
    - The compound score (a number between -1 and +1)
    """
    # polarity_scores() returns a dictionary like:
    # {'neg': 0.4, 'neu': 0.4, 'pos': 0.2, 'compound': -0.68}
    scores = analyzer.polarity_scores(text)
    compound = scores['compound']   # The most important score — overall emotion

    # Classify based on the compound score threshold
    if compound >= 0.05:
        sentiment = "POSITIVE"
    elif compound <= -0.05:
        sentiment = "NEGATIVE"
    else:
        sentiment = "NEUTRAL"

    return sentiment, round(compound, 3)


# Step 3: Run sentiment analysis on all posts
print("=" * 70)
print("SENTIMENT CLASSIFICATION RESULTS")
print("=" * 70)

sentiment_results = []  # Store results to use later in the combined pipeline

for post in posts:
    sentiment, score = get_sentiment(post["text"])
    sentiment_results.append(sentiment)

    # Shorten text for display (first 55 characters + "...")
    short_text = post["text"][:55] + "..."

    # Emoji to make it easier to read at a glance
    emoji = "😊" if sentiment == "POSITIVE" else ("😠" if sentiment == "NEGATIVE" else "😐")

    print(f"{emoji} [{sentiment:<8}] (score: {score:>6}) | True label: {post['label']:<7} | {short_text}")

print("=" * 70)
print("\n💡 Notice: Almost all 'stigma' posts are NEGATIVE sentiment!")
print("   This is why sentiment is a useful FIRST filter for stigma detection.")

SENTIMENT CLASSIFICATION RESULTS
😠 [NEGATIVE] (score: -0.791) | True label: stigma  | Schizophrenics are violent and dangerous. Keep them awa...
😠 [NEGATIVE] (score: -0.276) | True label: stigma  | Drug addicts chose this lifestyle. They deserve no symp...
😠 [NEGATIVE] (score: -0.765) | True label: stigma  | People with depression are just weak. They need to man ...
😠 [NEGATIVE] (score: -0.681) | True label: stigma  | Mental illness patients are a burden on the healthcare ...
😠 [NEGATIVE] (score: -0.625) | True label: stigma  | She is crazy. I would never hire someone with bipolar d...
😠 [NEGATIVE] (score: -0.784) | True label: stigma  | Alcoholics are just selfish people who ruin their famil...
😠 [NEGATIVE] (score: -0.638) | True label: stigma  | Why waste resources treating people who will never get ...
😊 [POSITIVE] (score:  0.188) | True label: support | Recovery is possible. You are not alone in this journey...
😊 [POSITIVE] (score:  0.735) | True label: support | People with mental

In [ ]:
# STEP 2 text classification
# ─────────────────────────────────────────────────────────────────────────────
# TEXT CLASSIFICATION using TF-IDF + Logistic Regression
# ─────────────────────────────────────────────────────────────────────────────

# ── STEP A: Split data into training set and test set ────────────────────────
# We train on 80% of the data and test on the remaining 20%
# test_size=0.2 means 20% goes to testing
# random_state=42 makes the split reproducible (same split every time)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

print(f"Training posts : {len(X_train)}")
print(f"Testing posts  : {len(X_test)}")


# ── STEP B: Convert text to numbers using TF-IDF ─────────────────────────────
# TF-IDF = Term Frequency × Inverse Document Frequency
#   - Words that appear often in ONE post but rarely in ALL posts get high scores
#   - Common words like 'the', 'is', 'and' get low scores
vectorizer = TfidfVectorizer(
    stop_words='english',  # Ignore very common English words ('the', 'a', 'is')
    ngram_range=(1, 2)     # Consider single words AND pairs of words (e.g., 'mental health')
)

# 'fit' = the vectorizer learns the vocabulary from training data
# 'transform' = it converts the text into a matrix of numbers
X_train_vec = vectorizer.fit_transform(X_train)  # Learn vocab AND convert
X_test_vec  = vectorizer.transform(X_test)       # Only convert (DO NOT re-learn)

print(f"\nVocabulary size (unique words/phrases): {len(vectorizer.vocabulary_)}")
print(f"Each post is now represented as a vector of {X_train_vec.shape[1]} numbers")


# ── STEP C: Train the Logistic Regression classifier ─────────────────────────
# Logistic Regression is a simple but powerful classification algorithm
# max_iter=1000 gives it enough time to find the best solution
classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(X_train_vec, y_train)   # 'fit' = learn from training data

print("\n✅ Model trained!")


# ── STEP D: Evaluate on the test set ─────────────────────────────────────────
y_pred = classifier.predict(X_test_vec)   # Predict labels for test posts
accuracy = accuracy_score(y_test, y_pred)

print("\n" + "=" * 55)
print("TEXT CLASSIFICATION — TEST RESULTS")
print("=" * 55)
print(f"\n📊 Overall Accuracy: {accuracy * 100:.1f}%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred,
                             target_names=["info", "stigma", "support"]))

print("\n📌 Predictions vs True Labels:")
for text, true, pred in zip(X_test, y_test, y_pred):
    match = "✅" if true == pred else "❌"
    print(f"  {match} True: {true:<8} | Predicted: {pred:<8} | {text[:50]}...")

Training posts : 16
Testing posts  : 4

Vocabulary size (unique words/phrases): 156
Each post is now represented as a vector of 156 numbers

✅ Model trained!

TEXT CLASSIFICATION — TEST RESULTS

📊 Overall Accuracy: 0.0%

Detailed Report:
              precision    recall  f1-score   support

        info       0.00      0.00      0.00       2.0
      stigma       0.00      0.00      0.00       2.0
     support       0.00      0.00      0.00       0.0

    accuracy                           0.00       4.0
   macro avg       0.00      0.00      0.00       4.0
weighted avg       0.00      0.00      0.00       4.0


📌 Predictions vs True Labels:
  ❌ True: stigma   | Predicted: support  | Schizophrenics are violent and dangerous. Keep the...
  ❌ True: info     | Predicted: support  | Sertraline is a commonly prescribed SSRI for anxie...
  ❌ True: info     | Predicted: support  | Schizophrenia affects about 1 in 300 people global...
  ❌ True: stigma   | Predicted: support  | Drug addicts cho

In [ ]:
#
# ─────────────────────────────────────────────────────────────────────────────
# Try it yourself! Predict the label of a brand-new post
# ─────────────────────────────────────────────────────────────────────────────

def classify_post(text):
    """
    Takes any new text and returns the predicted label.
    Uses the trained vectorizer + classifier.
    """
    # Convert the new text to numbers (using the SAME vectorizer — important!)
    vec = vectorizer.transform([text])
    # Predict the label
    prediction = classifier.predict(vec)[0]
    # Also get the probability for each class
    probs = classifier.predict_proba(vec)[0]
    class_names = classifier.classes_
    return prediction, dict(zip(class_names, [round(p, 3) for p in probs]))


# Test with 3 brand new posts the model has never seen
new_posts = [
    "Mental illness patients should be locked away from normal people.",
    "Everyone deserves support and access to mental healthcare.",
    "Anxiety disorders affect 1 in 13 people worldwide.",
]

print("=" * 60)
print("PREDICTING LABELS FOR NEW POSTS")
print("=" * 60)

for post in new_posts:
    label, probs = classify_post(post)
    emoji = {"stigma": "⚠️", "support": "💚", "info": "ℹ️"}.get(label, "")
    print(f"\n📝 Post   : {post}")
    print(f"   {emoji} Label  : {label.upper()}")
    print(f"   Probabilities: {probs}")

PREDICTING LABELS FOR NEW POSTS

📝 Post   : Mental illness patients should be locked away from normal people.
   ⚠️ Label  : STIGMA
   Probabilities: {np.str_('info'): np.float64(0.258), np.str_('stigma'): np.float64(0.437), np.str_('support'): np.float64(0.306)}

📝 Post   : Everyone deserves support and access to mental healthcare.
   💚 Label  : SUPPORT
   Probabilities: {np.str_('info'): np.float64(0.289), np.str_('stigma'): np.float64(0.327), np.str_('support'): np.float64(0.384)}

📝 Post   : Anxiety disorders affect 1 in 13 people worldwide.
   ℹ️ Label  : INFO
   Probabilities: {np.str_('info'): np.float64(0.362), np.str_('stigma'): np.float64(0.304), np.str_('support'): np.float64(0.335)}


In [ ]:
# Step 3 SEMANTICS ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
# SEMANTIC ANALYSIS using spaCy word vectors + cosine similarity
# ─────────────────────────────────────────────────────────────────────────────

# Load the spaCy English model with medium-sized word vectors
# 'en_core_web_md' contains 300-dimensional vectors for 20,000+ words
print("Loading spaCy model (this takes ~10 seconds)...")
nlp = spacy.load("en_core_web_md")
print("✅ spaCy model loaded!\n")


# ── STEP A: Define reference 'stigma sentences' ───────────────────────────────
# These are example sentences we KNOW are stigmatising.
# We will compare every post against these to find semantic matches.
stigma_references = [
    "mentally ill people are dangerous and violent criminals",
    "drug addicts are weak and worthless people",
    "people with mental illness should be isolated from society",
]

# Convert reference sentences to spaCy doc objects and get their vectors
# .vector gives us a numpy array of 300 numbers representing the sentence
ref_vectors = np.array([nlp(ref).vector for ref in stigma_references])


# ── STEP B: Define a function to check semantic stigma ────────────────────────
def semantic_stigma_score(text, threshold=0.65):
    """
    Compares input text against known stigma references.
    Returns:
        - max_sim  : highest similarity score found (0 to 1)
        - flagged  : True if score exceeds the threshold
        - best_ref : which reference sentence it matched most closely
    """
    # Get the vector for our input text
    text_vec = nlp(text).vector.reshape(1, -1)

    # Calculate cosine similarity against all reference vectors
    # cosine_similarity returns a matrix; we flatten it to a list
    similarities = cosine_similarity(text_vec, ref_vectors)[0]

    # Find the highest similarity score
    max_sim = float(np.max(similarities))
    best_ref_idx = int(np.argmax(similarities))

    flagged = max_sim >= threshold
    return round(max_sim, 3), flagged, stigma_references[best_ref_idx]


# ── STEP C: Run semantic analysis on all posts ────────────────────────────────
print("=" * 75)
print("SEMANTIC STIGMA DETECTION RESULTS")
print("=" * 75)
print(f"{'Post':<52} | {'Sim':>5} | {'Flagged':^9} | True Label")
print("-" * 75)

semantic_results = []

for post in posts:
    score, flagged, _ = semantic_stigma_score(post["text"])
    semantic_results.append(flagged)
    flag_str = "⚠️ STIGMA" if flagged else "✅ OK"
    short = post["text"][:50] + "..."
    print(f"{short:<53} | {score:>5} | {flag_str:<9} | {post['label']}")

print("=" * 75)


# ── STEP D: Show an example with a paraphrased stigma post ────────────────────
print("\n🔍 PARAPHRASED STIGMA EXAMPLE")
print("Can semantics catch stigma even when the words are different?\n")

paraphrased_tests = [
    "People with those disorders should not be allowed in public spaces.",  # Stigma phrased differently
    "I believe everyone deserves access to mental health treatment.",       # Supportive
    "The hospital runs free counselling sessions on Fridays.",             # Neutral info
]

for text in paraphrased_tests:
    score, flagged, matched_ref = semantic_stigma_score(text)
    verdict = "⚠️  STIGMA DETECTED" if flagged else "✅  No stigma detected"
    print(f"Post    : {text}")
    print(f"Result  : {verdict} (similarity: {score})")
    if flagged:
        print(f"Matched : '{matched_ref}'")
    print()

Loading spaCy model (this takes ~10 seconds)...
✅ spaCy model loaded!

SEMANTIC STIGMA DETECTION RESULTS
Post                                                 |   Sim |  Flagged  | True Label
---------------------------------------------------------------------------
Schizophrenics are violent and dangerous. Keep the... | 0.925 | ⚠️ STIGMA | stigma
Drug addicts chose this lifestyle. They deserve no... | 0.849 | ⚠️ STIGMA | stigma
People with depression are just weak. They need to... | 0.858 | ⚠️ STIGMA | stigma
Mental illness patients are a burden on the health... | 0.842 | ⚠️ STIGMA | stigma
She is crazy. I would never hire someone with bipo... | 0.803 | ⚠️ STIGMA | stigma
Alcoholics are just selfish people who ruin their ... | 0.879 | ⚠️ STIGMA | stigma
Why waste resources treating people who will never... | 0.851 | ⚠️ STIGMA | stigma
Recovery is possible. You are not alone in this jo... | 0.828 | ⚠️ STIGMA | support
People with mental health conditions deserve kindn... |  0.89 | ⚠️ S

In [ ]:
# step 4 NER
# ─────────────────────────────────────────────────────────────────────────────
# NAMED ENTITY RECOGNITION (NER) using spaCy
# ─────────────────────────────────────────────────────────────────────────────

# nlp is already loaded from Technique 3

# ── STEP A: A helper list of health-specific terms ────────────────────────────
# spaCy's default model knows PERSON, ORG, GPE etc.
# But it doesn't know medical terms, so we add our own lists

HEALTH_CONDITIONS = [
    "depression", "anxiety", "schizophrenia", "bipolar", "bipolar disorder",
    "addiction", "ocd", "adhd", "ptsd", "autism", "eating disorder",
    "mental illness", "psychosis", "dementia", "alzheimer"
]

HEALTH_DRUGS = [
    "sertraline", "fluoxetine", "prozac", "zoloft", "lithium",
    "ritalin", "adderall", "clonazepam", "valium", "xanax"
]


def extract_entities(text):
    """
    Extracts named entities from text using spaCy.
    Also checks for health conditions and drugs from our custom lists.

    Returns a list of (entity_text, entity_type) tuples.
    """
    doc = nlp(text)         # Process the text
    entities = []

    # ── Built-in spaCy entities (PERSON, ORG, GPE, etc.) ──────────────────────
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "FAC", "NORP"]:
            entities.append((ent.text, ent.label_))

    # ── Custom: Check for health conditions (lowercase comparison) ─────────────
    text_lower = text.lower()
    for condition in HEALTH_CONDITIONS:
        if condition in text_lower:
            # Find the original-casing from the text
            entities.append((condition, "HEALTH_CONDITION"))

    # ── Custom: Check for drugs ───────────────────────────────────────────────
    for drug in HEALTH_DRUGS:
        if drug in text_lower:
            entities.append((drug, "DRUG"))

    # Remove duplicates while preserving order
    seen = set()
    unique_entities = []
    for e in entities:
        if e not in seen:
            seen.add(e)
            unique_entities.append(e)

    return unique_entities


# ── STEP B: Run NER on some interesting posts ─────────────────────────────────
ner_test_posts = [
    "Dr. Mehta at AIIMS Delhi prescribed Sertraline for Priya's severe depression.",
    "John checked into a rehab centre in Mumbai after his opioid addiction diagnosis.",
    "Schizophrenics are violent and dangerous. Keep them away from society.",
    "WHO reports that depression affects over 280 million people worldwide.",
]

print("=" * 65)
print("NAMED ENTITY RECOGNITION RESULTS")
print("=" * 65)

for i, text in enumerate(ner_test_posts, 1):
    entities = extract_entities(text)
    print(f"\n📝 Post {i}: {text}")
    if entities:
        print("   Entities found:")
        for ent_text, ent_type in entities:
            icon = {
                "PERSON": "👤",
                "ORG": "🏥",
                "GPE": "📍",
                "HEALTH_CONDITION": "🩺",
                "DRUG": "💊",
            }.get(ent_type, "🔖")
            print(f"     {icon} [{ent_type:<18}] → '{ent_text}'")
    else:
        print("   No named entities found.")

print("\n" + "=" * 65)


# ── STEP C: Anonymise patient names (privacy protection) ──────────────────────
def anonymise(text):
    """
    Replaces all PERSON entities in the text with '[PATIENT]'.
    This is important for research datasets to protect privacy.
    """
    doc = nlp(text)
    anonymised = text
    # We replace from the end to avoid messing up character positions
    for ent in reversed(doc.ents):
        if ent.label_ == "PERSON":
            anonymised = anonymised[:ent.start_char] + "[PATIENT]" + anonymised[ent.end_char:]
    return anonymised


print("\n🔒 PATIENT ANONYMISATION DEMO")
original   = "Dr. Mehta prescribed Fluoxetine for Priya's anxiety disorder."
anonymised = anonymise(original)
print(f"  Original  : {original}")
print(f"  Anonymised: {anonymised}")

NAMED ENTITY RECOGNITION RESULTS

📝 Post 1: Dr. Mehta at AIIMS Delhi prescribed Sertraline for Priya's severe depression.
   Entities found:
     👤 [PERSON            ] → 'Mehta'
     🏥 [ORG               ] → 'AIIMS'
     📍 [GPE               ] → 'Delhi'
     👤 [PERSON            ] → 'Priya'
     🩺 [HEALTH_CONDITION  ] → 'depression'
     💊 [DRUG              ] → 'sertraline'

📝 Post 2: John checked into a rehab centre in Mumbai after his opioid addiction diagnosis.
   Entities found:
     👤 [PERSON            ] → 'John'
     📍 [GPE               ] → 'Mumbai'
     🩺 [HEALTH_CONDITION  ] → 'addiction'

📝 Post 3: Schizophrenics are violent and dangerous. Keep them away from society.
   No named entities found.

📝 Post 4: WHO reports that depression affects over 280 million people worldwide.
   Entities found:
     🩺 [HEALTH_CONDITION  ] → 'depression'


🔒 PATIENT ANONYMISATION DEMO
  Original  : Dr. Mehta prescribed Fluoxetine for Priya's anxiety disorder.
  Anonymised: Dr. [PATIENT] pre

In [ ]:
# COMBINED PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
# COMBINED PIPELINE — runs all 4 NLP techniques on any post
# ─────────────────────────────────────────────────────────────────────────────

def analyse_post(text, verbose=True):
    """
    Runs the complete social stigma detection pipeline on a single text post.

    Parameters:
        text    : The post to analyse (a string)
        verbose : If True, prints a detailed breakdown

    Returns:
        A dictionary with all results
    """

    # ── Technique 1: Sentiment ─────────────────────────────────────────────────
    sentiment, sent_score = get_sentiment(text)

    # ── Technique 2: Text Classification ──────────────────────────────────────
    clf_label, clf_probs = classify_post(text)

    # ── Technique 3: Semantics ─────────────────────────────────────────────────
    sem_score, sem_flagged, sem_ref = semantic_stigma_score(text)

    # ── Technique 4: NER ──────────────────────────────────────────────────────
    entities = extract_entities(text)
    anonymised_text = anonymise(text)

    # ── Combine into a Stigma Risk Score ──────────────────────────────────────
    risk_score = 0
    if sentiment == "NEGATIVE":     risk_score += 1   # Weak signal
    if clf_label == "stigma":       risk_score += 2   # Strong signal
    if sem_flagged:                 risk_score += 2   # Strong signal

    if   risk_score >= 4: risk_level = "🔴 HIGH RISK — Likely stigmatising"
    elif risk_score >= 2: risk_level = "🟡 MEDIUM RISK — Possible stigma"
    else:                 risk_level = "🟢 LOW RISK — Likely safe"

    # ── Print the detailed report ──────────────────────────────────────────────
    if verbose:
        print("\n" + "═" * 65)
        print("  SOCIAL STIGMA DETECTION — FULL ANALYSIS")
        print("═" * 65)
        print(f"  📝 Post      : {text}")
        print("─" * 65)
        print(f"  1️⃣  Sentiment    : {sentiment} (compound score: {sent_score})")
        print(f"  2️⃣  Classification: {clf_label.upper()} | Probabilities: {clf_probs}")
        print(f"  3️⃣  Semantics     : Similarity = {sem_score} | Flagged: {sem_flagged}")
        if sem_flagged:
            print(f"     ↳ Matched reference: '{sem_ref}'")
        print(f"  4️⃣  NER           : {entities if entities else 'No entities found'}")
        print(f"     ↳ Anonymised  : {anonymised_text}")
        print("─" * 65)
        print(f"  🎯 STIGMA RISK SCORE  : {risk_score}/5")
        print(f"  📌 VERDICT            : {risk_level}")
        print("═" * 65)

    return {
        "text": text, "sentiment": sentiment, "sentiment_score": sent_score,
        "classification": clf_label, "semantic_flagged": sem_flagged,
        "semantic_score": sem_score, "entities": entities,
        "risk_score": risk_score, "risk_level": risk_level,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Test the combined pipeline on 4 sample posts
# ─────────────────────────────────────────────────────────────────────────────

test_cases = [
    # Clearly stigmatising
    "People with schizophrenia are violent and should be locked away from the public.",
    # Supportive
    "Everyone struggling with mental illness deserves compassion and quality healthcare.",
    # Informational
    "NIMHANS in Bengaluru offers residential treatment programmes for severe depression.",
    # Subtle stigma (paraphrased — tests semantic detection)
    "Honestly, those kinds of people don't belong in normal workplaces.",
]

all_results = []
for post in test_cases:
    result = analyse_post(post, verbose=True)
    all_results.append(result)


═════════════════════════════════════════════════════════════════
  SOCIAL STIGMA DETECTION — FULL ANALYSIS
═════════════════════════════════════════════════════════════════
  📝 Post      : People with schizophrenia are violent and should be locked away from the public.
─────────────────────────────────────────────────────────────────
  1️⃣  Sentiment    : NEGATIVE (compound score: -0.599)
  2️⃣  Classification: STIGMA | Probabilities: {np.str_('info'): np.float64(0.272), np.str_('stigma'): np.float64(0.366), np.str_('support'): np.float64(0.362)}
  3️⃣  Semantics     : Similarity = 0.881 | Flagged: True
     ↳ Matched reference: 'mentally ill people are dangerous and violent criminals'
  4️⃣  NER           : [('schizophrenia', 'HEALTH_CONDITION')]
     ↳ Anonymised  : People with schizophrenia are violent and should be locked away from the public.
─────────────────────────────────────────────────────────────────
  🎯 STIGMA RISK SCORE  : 5/5
  📌 VERDICT            : 🔴 HIGH RISK — Like